# Analyzing the SIDER 4.1 data

+ http://sideeffects.embl.de/
+ http://thinklab.com/d/30#4

In [4]:
import csv
import gzip
import collections

In [5]:
import pandas

In [8]:
# Download SIDER data
base_url = 'http://sideeffects.embl.de/media/download/'
filenames = [
    'README',
    'meddra_all_indications.tsv.gz',
    'meddra_all_se.tsv.gz',
    'meddra_freq.tsv.gz',
]
for filename in filenames:
    ! wget --no-verbose --timestamping --directory-prefix download {base_url}/{filename}

! mv download/README download/README.txt

2024-09-04 11:56:48 URL:http://sideeffects.embl.de/media/download//README [3304/3304] -> "download/README" [1]


## STITCH to DrugBank mapping utilities

In [7]:
def stitch_flat_to_pubchem(cid):
    assert cid.startswith('CID')
    return int(cid[3:]) - 1e8

def stitch_stereo_to_pubchem(cid):
    assert cid.startswith('CID')
    return int(cid[3:])

In [9]:
# Read DrugBank terms
url = 'https://raw.githubusercontent.com/dhimmel/drugbank/3e87872db5fca5ac427ce27464ab945c0ceb4ec6/data/drugbank.tsv'
drugbank_df = pandas.read_table(url)[['drugbank_id', 'name']].rename(columns={'name': 'drugbank_name'})

# Pubchem to DrugBank mapping
url = 'https://raw.githubusercontent.com/dhimmel/drugbank/3e87872db5fca5ac427ce27464ab945c0ceb4ec6/data/mapping/pubchem.tsv'
drugbank_map_df = pandas.read_table(url)

In [12]:
url = 'https://raw.githubusercontent.com/iit-Demokritos/drug_id_mapping/main/drug-mappings.tsv'
drugbank_map_df_2 = pandas.read_table(url)

In [13]:
drugbank_map_df_2

,drugbankId,name,ttd_id,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
0,DB13088,AZD-0424,D0QG8F,9893171.0,692054-06-1,CHEMBL3545177,NaN,NaN,NaN,NaN,NaN,C4519307,NaN
1,DB13089,Enoxolone,D06EWG,10114.0,471-53-4,CHEMBL230006,ZINC000019203131,30853,C02283,NaN,50233538.0,C0017986,NaN
2,DB13082,Nefiracetam,D0KD5P,71157.0,77191-36-7,CHEMBL260829,ZINC000000003788,135004,NaN,NaN,NaN,C0165264,NaN
3,DB13083,Talarozole,D0AN7B,9799888.0,201410-53-9,CHEMBL459505,NaN,102167,NaN,D09385,50253810.0,C2606129,NaN
4,DB13080,Roluperidone,D0SQ1W,9799284.0,359625-79-9,NaN,NaN,NaN,NaN,NaN,NaN,C4730997,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22344,NaN,N-(2-amino-5-(thiophen-2-yl)phenyl)nicotinamide,D0Q2GM,44454416.0,NaN,CHEMBL256440,ZINC000029127438,NaN,NaN,NaN,50232034.0,NaN,NaN
22345,NaN,2-Cinnamamido-N1-hydroxy-N4-pentylsuccinamide,D0F2HJ,45257111.0,NaN,CHEMBL596560,ZINC000045358109,NaN,NaN,NaN,50304541.0,NaN,NaN
22346,NaN,4-Biphenyl-4-yl-2-cyclohexylmethyl-1H-imidazole,D08KBV,9966703.0,NaN,CHEMBL334104,ZINC000027106397,NaN,NaN,NaN,50223722.0,NaN,NaN
22347,NaN,Naldemedine Tosylate,D0Q2HO,56837137.0,1345728-04-2,CHEMBL3039508,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
drugbank_map_df_2[drugbank_map_df_2['drugbankId'] == 'DB00115']

,drugbankId,name,ttd_id,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
2318,DB00115,Cyanocobalamin,D0X4NX,70678590.0,68-19-9,CHEMBL2110563,NaN,17439,C02823,D00166,NaN,C0042845,CID100002891


In [6]:
drugbank_map_df

,drugbank_id,pubchem_id
0,DB00014,11980055
1,DB00014,11981235
2,DB00014,11982741
3,DB00014,16052011
4,DB00014,23581804
...,...,...
204704,DB09028,74070157
204705,DB09028,74834862
204706,DB09028,77513518
204707,DB09028,87355970


## meddra_freq.tsv.gz

In [7]:
columns = [
    'stitch_id_flat',
    'stitch_id_sterio',
    'umls_cui_from_label',
    'placebo',
    'frequency',
    'lower',
    'upper',
    'meddra_type',
    'umls_cui_from_meddra',
    'side_effect_name',
]
freq_df = pandas.read_table('download/meddra_freq.tsv.gz', names=columns)
freq_df.head(2)

,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,placebo,frequency,lower,upper,meddra_type,umls_cui_from_meddra,side_effect_name
0,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,LLT,C0000737,Abdominal pain
1,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,PT,C0000737,Abdominal pain


## meddra_all_se.tsv.gz

In [9]:
columns = [
    'stitch_id_flat',
    'stitch_id_sterio',
    'umls_cui_from_label',
    'meddra_type',
    'umls_cui_from_meddra',
    'side_effect_name',
]
se_df = pandas.read_table('download/meddra_all_se.tsv.gz', names=columns)
print(se_df.shape)
se_df['pubchem_id'] = se_df.stitch_id_sterio.map(stitch_stereo_to_pubchem)
print(se_df.shape)
#se_df = drugbank_map_df.merge(se_df)
print(se_df.shape)
se_df.head(2)

(309849, 6)
(309849, 7)
(309849, 7)


,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name,pubchem_id
0,CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps,10917
1,CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain,10917


In [11]:
se_df[se_df['pubchem_id'] == 143]

,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name,pubchem_id
322,CID100000143,CID000000143,C0002170,LLT,C0002170,Alopecia,143
323,CID100000143,CID000000143,C0002170,PT,C0002170,Alopecia,143
324,CID100000143,CID000000143,C0003123,LLT,C0003123,Anorexia,143
325,CID100000143,CID000000143,C0003123,PT,C0232462,Decreased appetite,143
326,CID100000143,CID000000143,C0009450,LLT,C0009450,Infection,143
327,CID100000143,CID000000143,C0009450,PT,C0009450,Infection,143
328,CID100000143,CID000000143,C0009806,LLT,C0009806,Constipation,143
329,CID100000143,CID000000143,C0009806,PT,C0009806,Constipation,143
330,CID100000143,CID000000143,C0011603,LLT,C0011603,Dermatitis,143
331,CID100000143,CID000000143,C0011603,PT,C0011603,Dermatitis,143


In [10]:
se_df.pubchem_id.map(pubchem_id_to_drug_name_unichem)

143


UnboundLocalError: cannot access local variable 'content' where it is not associated with a value

In [56]:
drugbank_df[drugbank_df['drugbank_id'] == 'DB08875']

,drugbank_id,drugbank_name
7617,DB08875,Cabozantinib


In [65]:
drugbank_map_df[drugbank_map_df['drugbank_id'] == "DB08875"]

,drugbank_id,pubchem_id


In [110]:
help(pandas.DataFrame.merge)

Help on function merge in module pandas.core.frame:

merge(self, right: 'DataFrame | Series', how: 'MergeHow' = 'inner', on: 'IndexLabel | AnyArrayLike | None' = None, left_on: 'IndexLabel | AnyArrayLike | None' = None, right_on: 'IndexLabel | AnyArrayLike | None' = None, left_index: 'bool' = False, right_index: 'bool' = False, sort: 'bool' = False, suffixes: 'Suffixes' = ('_x', '_y'), copy: 'bool | None' = None, indicator: 'str | bool' = False, validate: 'MergeValidate | None' = None) -> 'DataFrame'
    Merge DataFrame or named Series objects with a database-style join.

    A named Series object is treated as a DataFrame with a single named column.

    The join is done on columns or indexes. If joining columns on
    columns, the DataFrame indexes *will be ignored*. Otherwise if joining indexes
    on indexes or indexes on a column or columns, the index will be passed on.
    When performing a cross merge, no column specifications to merge on are
    allowed.

    .. warning::

    

In [117]:
# se_df['drugbank_id'] = 
se_df.pubchem_id.map(pubchem_id_to_drug_name_unichem)

143


UnboundLocalError: cannot access local variable 'content' where it is not associated with a value

In [60]:
se_df = se_df.merge(drugbank_df, how='left')
# se_df = se_df[['drugbank_id', 'umls_cui_from_meddra', 'side_effect_name']]
se_df

MergeError: No common columns to perform merge on. Merge options: left_on=None, right_on=None, left_index=False, right_index=False

In [25]:
se_df[se_df['umls_cui_from_meddra'] == 'C1145670'].sort_values(by='drugbank_id').drop

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
35765,DB00181,2284,CID100002284,CID000002284,C3203358,PT,C1145670,Respiratory failure
15117,DB00184,89594,CID100000942,CID000089594,C1145670,PT,C1145670,Respiratory failure
15116,DB00184,89594,CID100000942,CID000089594,C1145670,LLT,C1145670,Respiratory failure
123563,DB00186,3958,CID100003958,CID000003958,C3203358,PT,C1145670,Respiratory failure
123554,DB00186,3958,CID100003958,CID000003958,C1145670,LLT,C1145670,Respiratory failure
...,...,...,...,...,...,...,...,...
348325,NaN,25102847,CID125102847,CID025102847,C1145670,PT,C1145670,Respiratory failure
351931,NaN,70683024,CID170683024,CID070683024,C1145670,LLT,C1145670,Respiratory failure
351932,NaN,70683024,CID170683024,CID070683024,C1145670,PT,C1145670,Respiratory failure
352629,NaN,71306834,CID171306834,CID071306834,C0035229,PT,C1145670,Respiratory failure


In [17]:
se_df = se_df.dropna()
se_df

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
0,DB00583,10917,CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps
1,DB02648,10917,CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps
2,DB00583,10917,CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain
3,DB02648,10917,CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain
4,DB00583,10917,CID100000085,CID000010917,C0000737,LLT,C0000737,Abdominal pain
...,...,...,...,...,...,...,...,...
351857,DB00159,56842239,CID156842239,CID056842239,C0162429,LLT,C0162429,Malnutrition
351858,DB03756,56842239,CID156842239,CID056842239,C0162429,LLT,C0162429,Malnutrition
351859,DB00132,56842239,CID156842239,CID056842239,C0162429,PT,C0162429,Malnutrition
351860,DB00159,56842239,CID156842239,CID056842239,C0162429,PT,C0162429,Malnutrition


In [49]:
se_df[(se_df['umls_cui_from_meddra'] == 'C1145670') & (se_df['pubchem_id'] == 25102847)]

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
348324,NaN,25102847,CID125102847,CID025102847,C1145670,LLT,C1145670,Respiratory failure
348325,NaN,25102847,CID125102847,CID025102847,C1145670,PT,C1145670,Respiratory failure


In [26]:
side_effect_MS = se_df[se_df['umls_cui_from_meddra'] == 'C1145670'].query("meddra_type == 'PT'")

In [43]:
side_effect_MS.merge(drugbank_df).drop_duplicates(subset='drugbank_id').sort_values(['drugbank_name']).drop_duplicates(subset=['drugbank_name'])['drugbank_name'].to_csv("drugbank_name.csv")

In [44]:
side_effect_MS['drugbank_id']

359       DB00855
1957      DB00194
1958      DB00640
1959      DB03528
2116      DB02379
           ...   
337755    DB08865
348325        NaN
351932        NaN
352629        NaN
353109        NaN
Name: drugbank_id, Length: 194, dtype: object

In [92]:
side_effect_MS.columns

Index(['stitch_id_flat', 'stitch_id_sterio', 'umls_cui_from_label',
       'meddra_type', 'umls_cui_from_meddra', 'side_effect_name',
       'pubchem_id'],
      dtype='object')

In [99]:
side_effect_MS['name'] = side_effect_MS.apply(stitch_id_to_drug_name, axis=1)

[{"taxonName":"small molecule","queryIndex":0,"ncbiTaxonId":-1,"annotation":"cevimeline","preferredName":"cevimeline","stringId":"-1.CID100002684"}]
[{"taxonName":"small molecule","annotation":"imiquimod","ncbiTaxonId":-1,"queryIndex":0,"stringId":"-1.CID100057469","preferredName":"imiquimod"}]
[{"preferredName":"pregabalin","ncbiTaxonId":-1,"annotation":"pregabalin","queryIndex":0,"stringId":"-1.CID100125889","taxonName":"small molecule"}]
[{"annotation":"varenicline","taxonName":"small molecule","preferredName":"varenicline","queryIndex":0,"ncbiTaxonId":-1,"stringId":"-1.CID100170361"}]
<!DOCTYPE html>

<head>
<meta http-equiv='Content-type' content='text/html;charset=UTF-8' />
<link rel='shortcut icon' href='/images/favicon.png' />
<link href='https://fonts.googleapis.com/css?family=Open+Sans:400,300,600,700,800' rel='stylesheet' type='text/css'/>
<link href='https://fonts.googleapis.com/css?family=Raleway:400,300,500,700' rel='stylesheet' type='text/css'/>
<title>STITCH: chemical a

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

### DataFrame Functions: Cross Reference Other ID formats Given pubchem_id

In [51]:
def pubchem_id_to_drug_name(cid):
    import requests

    # cid = str(row['pubchem_id'])  # Replace with your CID
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
    
    response = requests.get(url)
    data = response.json()
    first_synonym = data['InformationList']['Information'][0]['Synonym'][0]

    return first_synonym

In [79]:
import requests
# source_compound_id = 25102847
url = f"https://www.ebi.ac.uk/unichem/api/v1/compounds"
parameter = {
          "compound": "25102847",
          "sourceID": 22,
          "type": "sourceID"
        }
response = requests.get(url, params=parameter)
data = response.json()

In [2]:
def pubchem_id_to_drug_name_unichem(uci):
    url = "https://www.ebi.ac.uk/unichem/api/v1/compounds"
    data = {
        "compound": uci,
        "sourceID": 22,
        "type": "sourceID"
    }
    
    response = requests.post(url, json=data)
    if response.status_code == 200:
        try:
            content = response.json()['compounds'][0]['sources']
        except:
            print(uci)
        for i in content:
            if i['id'] == 45:
                name = [i['compoundId']]
                return name[0]
        return None
    else:
        return None


In [106]:
pubchem_id_to_drug_name_unichem("25102847")

'CABOZANTINIB'

In [83]:
import requests

url = "https://www.ebi.ac.uk/unichem/api/v1/compounds"
# headers = {
#     "accept": "application/json",
#     "Content-Type": "application/json"
# }
data = {
    "compound": "25102847",
    "sourceID": 22,
    "type": "sourceID"
}

response = requests.post(url, json=data)


In [104]:
content = response.json()['compounds'][0]['sources']
# for i in content:
#     if i['id'] == 45:
#         print(i['compoundId'])
print([i['compoundId'] for i in content if i['id'] == 45])


['CABOZANTINIB']


#### Some InChIKey doesn't have corresponding ChEMBL ID

In [73]:
def pubchem_id_to_InChIKey_to_ChEMBL_ID(row):
    import requests

    cid = int(row['pubchem_id'])  # Replace with your CID
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/InChI,InChIKey/JSON"

    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
    else:
        print(cid)
    try:
        InChIKey = data['PropertyTable']['Properties'][0]['InChIKey']
    except KeyError:
        print(data)
    # print(InChIKey)
    url = f"https://www.ebi.ac.uk/unichem/rest/inchikey/{InChIKey}"
    response = requests.get(url)
    data = response.json()

    Unichem_df = pd.DataFrame(data)
    if '1' in Unichem_df['src_id'].to_list():
        chembl_id_object = Unichem_df[Unichem_df['src_id'] == '1']['src_compound_id']
    # print(Unichem_df)
        chembl_id = chembl_id_object.values[0]
    else:
        chembl_id = None
    return chembl_id

In [29]:
side_effect_MS['drug_name'] = side_effect_MS.apply(pubchem_id_to_drug_name, axis=1)
side_effect_MS

,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name,pubchem_id,drug_name
38739,CID100002684,CID000083898,C0026769,PT,C0026769,Multiple sclerosis,83898,Cevimeline
211095,CID100057469,CID000057469,C0026769,PT,C0026769,Multiple sclerosis,57469,IMIQUIMOD
216046,CID100060754,CID000060754,C0026769,PT,C0026769,Multiple sclerosis,60754,Gadodiamide hydrate
247264,CID100125889,CID005486971,C0026769,PT,C0026769,Multiple sclerosis,5486971,Pregabalin
259034,CID100170361,CID000170361,C0026769,PT,C0026769,Multiple sclerosis,170361,Varenicline
299280,CID116131310,CID016131310,C0026769,PT,C0026769,Multiple sclerosis,16131310,Heplisav
307569,CID154677977,CID054684141,C0026769,PT,C0026769,Multiple sclerosis,54684141,Teriflunomide


#### For unavailable ChEMBL IDs, we can either impute manually or use code snippet from ATC to ChEMBL jupyter notebook to search the drug name in ChEMBL.

In [48]:
side_effect_MS['ChEMBL_ID'] = side_effect_MS.apply(pubchem_id_to_InChIKey_to_ChEMBL_ID, axis=1)
side_effect_MS

,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name,pubchem_id,drug_name,ChEMBL_ID
38739,CID100002684,CID000083898,C0026769,PT,C0026769,Multiple sclerosis,83898,Cevimeline,CHEMBL1201267
211095,CID100057469,CID000057469,C0026769,PT,C0026769,Multiple sclerosis,57469,IMIQUIMOD,CHEMBL1282
216046,CID100060754,CID000060754,C0026769,PT,C0026769,Multiple sclerosis,60754,Gadodiamide hydrate,None
247264,CID100125889,CID005486971,C0026769,PT,C0026769,Multiple sclerosis,5486971,Pregabalin,CHEMBL1059
259034,CID100170361,CID000170361,C0026769,PT,C0026769,Multiple sclerosis,170361,Varenicline,CHEMBL1396
299280,CID116131310,CID016131310,C0026769,PT,C0026769,Multiple sclerosis,16131310,Heplisav,None
307569,CID154677977,CID054684141,C0026769,PT,C0026769,Multiple sclerosis,54684141,Teriflunomide,CHEMBL973


In [9]:
import requests

cid = "54684141"  # Replace with your CID
url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"

response = requests.get(url)
data = response.json()
synonym_first = data['InformationList']['Information'][0]['Synonym'][0]

'Teriflunomide'

In [11]:
import requests

cid = "54684141"  # Replace with your CID
url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/InChI,InChIKey/JSON"

response = requests.get(url)
data = response.json()
data['PropertyTable']['Properties'][0]['InChIKey']
# Example: print out the compound's name and molecular formula
# print("Name:", data['PropertyTable']['Properties'][0]['IUPACName'])
# print("Molecular Formula:", data['PropertyTable']['Properties'][0]['MolecularFormula'])


'UTNUDOFZCWSZMS-YFHOEESVSA-N'

In [12]:
import requests

InChIKey = 'UTNUDOFZCWSZMS-YFHOEESVSA-N'
url = f"https://www.ebi.ac.uk/unichem/rest/inchikey/{InChIKey}"

response = requests.get(url)

In [13]:
data = response.json()


In [14]:
data

[{'src_id': '7', 'src_compound_id': '68540'},
 {'src_id': '14', 'src_compound_id': '1C058IKG3B'},
 {'src_id': '21', 'src_compound_id': '15221873'},
 {'src_id': '22', 'src_compound_id': '54684141'},
 {'src_id': '49', 'src_compound_id': 'PD010419'},
 {'src_id': '26', 'src_compound_id': '108605-62-5'},
 {'src_id': '3', 'src_compound_id': 'A26'},
 {'src_id': '15', 'src_compound_id': 'SCHEMBL22661'},
 {'src_id': '9', 'src_compound_id': 'ZINC000013512456'},
 {'src_id': '39', 'src_compound_id': 'CB5648113'},
 {'src_id': '40', 'src_compound_id': 'teriflunomide'},
 {'src_id': '32', 'src_compound_id': 'DTXSID80893457'},
 {'src_id': '39', 'src_compound_id': 'CB02486749'},
 {'src_id': '39', 'src_compound_id': 'CB6464433'},
 {'src_id': '2', 'src_compound_id': 'DB08880'},
 {'src_id': '37', 'src_compound_id': '55910'},
 {'src_id': '37', 'src_compound_id': '163092'},
 {'src_id': '37', 'src_compound_id': '6758'},
 {'src_id': '37', 'src_compound_id': '147806'},
 {'src_id': '36', 'src_compound_id': 'MTBL

In [4]:
import pandas as pd

In [33]:
Unichem_df = pd.DataFrame(data)
chembl_id = Unichem_df[Unichem_df['src_id'] == '1']['src_compound_id']
chembl_id.values[0]

'CHEMBL973'

In [116]:
data

{'InformationList': {'Information': [{'CID': 54684141,
    'Synonym': ['Teriflunomide',
     '163451-81-8',
     '108605-62-5',
     'Aubagio',
     'Flucyamide',
     'A77 1726',
     'HMR1726',
     '(Z)-2-Cyano-3-hydroxy-N-(4-(trifluoromethyl)phenyl)but-2-enamide',
     'HMR 1726',
     'HMR-1726',
     'SU 20',
     'A 771726',
     'A-771726',
     'A771726',
     'teriflunomida',
     'teriflunomidum',
     'A 77-1726',
     'CHEBI:68540',
     'UNII-1C058IKG3B',
     '1C058IKG3B',
     '2-cyano-3-hydroxy-N-(4-(trifluoromethyl)phenyl)but-2-enamide',
     '(E/Z)-Teriflunomide',
     '(Z)-2-cyano-3-hydroxy-N-[4-(trifluoromethyl)phenyl]but-2-enamide',
     '(Z)-2-cyano-alpha,alpha,alpha-trifluoro-3-hydroxy-p-crotonotoluidide',
     'N-(4-Trifluoromethylphenyl)-2-cyano-2-hydroxycrotonamide',
     '2-hydroxyethylidene-cyanoacetic acid-4-trifluoromethyl anilide',
     'A 1726',
     '(2z)-2-Cyano-3-Hydroxy-N-[4-(Trifluoromethyl)phenyl]but-2-Enamide',
     '2-Butenamide, 2-cyano-3-hydro

In [10]:
se_df[se_df['umls_cui_from_label'] == 'C0235431']

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
4206,DB00177,60846,CID100005650,CID000060846,C0235431,LLT,C0235431,Blood creatinine increased
4207,DB00177,60846,CID100005650,CID000060846,C0235431,PT,C0235431,Blood creatinine increased
9818,DB00193,33741,CID100005523,CID000033741,C0235431,LLT,C0235431,Blood creatinine increased
9819,DB00193,33741,CID100005523,CID000033741,C0235431,PT,C0235431,Blood creatinine increased
17910,DB00218,101526,CID100004259,CID000101526,C0235431,LLT,C0235431,Blood creatinine increased
...,...,...,...,...,...,...,...,...
297492,DB08889,11556711,CID111556711,CID011556711,C0235431,PT,C0235431,Blood creatinine increased
298657,DB08907,24812758,CID124812758,CID024812758,C0235431,LLT,C0235431,Blood creatinine increased
298658,DB08907,24812758,CID124812758,CID024812758,C0235431,PT,C0235431,Blood creatinine increased
298917,DB08910,134780,CID100134780,CID000134780,C0235431,LLT,C0235431,Blood creatinine increased


In [11]:
se_df[(se_df['umls_cui_from_label'] == 'C0235431') & (se_df['meddra_type'] == 'PT')]

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
4207,DB00177,60846,CID100005650,CID000060846,C0235431,PT,C0235431,Blood creatinine increased
9819,DB00193,33741,CID100005523,CID000033741,C0235431,PT,C0235431,Blood creatinine increased
17911,DB00218,101526,CID100004259,CID000101526,C0235431,PT,C0235431,Blood creatinine increased
19090,DB00225,60754,CID100060754,CID000060754,C0235431,PT,C0235431,Blood creatinine increased
20676,DB00230,5486971,CID100125889,CID005486971,C0235431,PT,C0235431,Blood creatinine increased
...,...,...,...,...,...,...,...,...
294191,DB08816,9871419,CID109871419,CID009871419,C0235431,PT,C0235431,Blood creatinine increased
294464,DB08822,11238823,CID111238823,CID011238823,C0235431,PT,C0235431,Blood creatinine increased
297492,DB08889,11556711,CID111556711,CID011556711,C0235431,PT,C0235431,Blood creatinine increased
298658,DB08907,24812758,CID124812758,CID024812758,C0235431,PT,C0235431,Blood creatinine increased


In [12]:
se_df[(se_df['umls_cui_from_label'] == 'C0235431') & (se_df['meddra_type'] == 'LLT')]

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
4206,DB00177,60846,CID100005650,CID000060846,C0235431,LLT,C0235431,Blood creatinine increased
9818,DB00193,33741,CID100005523,CID000033741,C0235431,LLT,C0235431,Blood creatinine increased
17910,DB00218,101526,CID100004259,CID000101526,C0235431,LLT,C0235431,Blood creatinine increased
19089,DB00225,60754,CID100060754,CID000060754,C0235431,LLT,C0235431,Blood creatinine increased
20675,DB00230,5486971,CID100125889,CID005486971,C0235431,LLT,C0235431,Blood creatinine increased
...,...,...,...,...,...,...,...,...
294190,DB08816,9871419,CID109871419,CID009871419,C0235431,LLT,C0235431,Blood creatinine increased
294463,DB08822,11238823,CID111238823,CID011238823,C0235431,LLT,C0235431,Blood creatinine increased
297491,DB08889,11556711,CID111556711,CID011556711,C0235431,LLT,C0235431,Blood creatinine increased
298657,DB08907,24812758,CID124812758,CID024812758,C0235431,LLT,C0235431,Blood creatinine increased


In [13]:
se_df[se_df['umls_cui_from_meddra'] == 'C0235431']

,drugbank_id,pubchem_id,stitch_id_flat,stitch_id_sterio,umls_cui_from_label,meddra_type,umls_cui_from_meddra,side_effect_name
1362,DB00091,5280754,CID100002909,CID005280754,C0151578,PT,C0235431,Blood creatinine increased
1506,DB00091,5280754,CID100002909,CID005280754,C0700225,PT,C0235431,Blood creatinine increased
2587,DB00136,5280453,CID100002524,CID005280453,C0700225,PT,C0235431,Blood creatinine increased
4206,DB00177,60846,CID100005650,CID000060846,C0235431,LLT,C0235431,Blood creatinine increased
4207,DB00177,60846,CID100005650,CID000060846,C0235431,PT,C0235431,Blood creatinine increased
...,...,...,...,...,...,...,...,...
298917,DB08910,134780,CID100134780,CID000134780,C0235431,LLT,C0235431,Blood creatinine increased
298918,DB08910,134780,CID100134780,CID000134780,C0235431,PT,C0235431,Blood creatinine increased
299127,DB08911,11707110,CID111707110,CID011707110,C0151578,PT,C0235431,Blood creatinine increased
299417,DB08912,44462760,CID144462760,CID044462760,C0151578,PT,C0235431,Blood creatinine increased


In [14]:
se_df = se_df[['drugbank_id', 'umls_cui_from_meddra', 'side_effect_name']]
se_df = se_df.dropna()
se_df = se_df.drop_duplicates(['drugbank_id', 'umls_cui_from_meddra'])
se_df = drugbank_df.merge(se_df)
se_df = se_df.sort_values(['drugbank_name', 'side_effect_name'])
len(se_df)

153663

In [15]:
se_df

,drugbank_id,drugbank_name,umls_cui_from_meddra,side_effect_name
146269,DB07768,"(10ALPHA,13ALPHA,14BETA,17ALPHA)-17-HYDROXYAND...",C0000729,Abdominal cramps
146270,DB07768,"(10ALPHA,13ALPHA,14BETA,17ALPHA)-17-HYDROXYAND...",C0000737,Abdominal pain
146524,DB07768,"(10ALPHA,13ALPHA,14BETA,17ALPHA)-17-HYDROXYAND...",C0232492,Abdominal pain upper
146622,DB07768,"(10ALPHA,13ALPHA,14BETA,17ALPHA)-17-HYDROXYAND...",C0740651,Abdominal symptom
146645,DB07768,"(10ALPHA,13ALPHA,14BETA,17ALPHA)-17-HYDROXYAND...",C0877331,Abnormal clotting factor
...,...,...,...,...
138639,DB05738,vapitadine dihydrochloride,C0015230,Rash
138654,DB05738,vapitadine dihydrochloride,C0234233,Tenderness
138650,DB05738,vapitadine dihydrochloride,C0041582,Ulcer
138651,DB05738,vapitadine dihydrochloride,C0042487,Venous thrombosis


In [16]:
# Create a reference of side effect IDs and Names
se_terms_df = se_df[['umls_cui_from_meddra', 'side_effect_name']].drop_duplicates()
assert se_terms_df.side_effect_name.duplicated().sum() == 0
se_terms_df = se_terms_df.sort_values('side_effect_name')
se_terms_df.to_csv('side-effect-terms.tsv', sep='\t', index=False)

In [17]:
se_df[se_df['side_effect_name'] == 'Multiple sclerosis']

,drugbank_id,drugbank_name,umls_cui_from_meddra,side_effect_name
3214,DB00185,Cevimeline,C0026769,Multiple sclerosis
9584,DB00225,Gadodiamide,C0026769,Multiple sclerosis
63677,DB00724,Imiquimod,C0026769,Multiple sclerosis
10096,DB00230,Pregabalin,C0026769,Multiple sclerosis
120226,DB01273,Varenicline,C0026769,Multiple sclerosis


In [18]:
# Side effects of cocaine
se_df.query("drugbank_id == 'DB00953'")

,drugbank_id,drugbank_name,umls_cui_from_meddra,side_effect_name
84564,DB00953,Rizatriptan,C0232487,Abdominal discomfort
84445,DB00953,Rizatriptan,C0000731,Abdominal distension
84611,DB00953,Rizatriptan,C0000737,Abdominal pain
84507,DB00953,Rizatriptan,C0948089,Acute coronary syndrome
84548,DB00953,Rizatriptan,C0085631,Agitation
...,...,...,...,...
84540,DB00953,Rizatriptan,C0042571,Vertigo
84582,DB00953,Rizatriptan,C0344232,Vision blurred
84541,DB00953,Rizatriptan,C0042963,Vomiting
84542,DB00953,Rizatriptan,C0043144,Wheezing


In [19]:
# Number of drugbank drugs
se_df.drugbank_id.nunique()

1223

In [20]:
# Number of UMLS side effects
se_df.umls_cui_from_meddra.nunique()

5734

In [21]:
# Save side effects
# se_df.to_csv('data/side-effects.tsv', sep='\t', index=False)

## meddra_all_indications.tsv.gz

In [60]:
columns = [
    'stitch_id_flat',
    'umls_cui_from_label',
    'method',
    'concept_name',
    'meddra_type',
    'umls_cui_from_meddra',
    'meddra_name',
]
indication_df = pandas.read_table('download/meddra_all_indications.tsv.gz', names=columns)
indication_df['pubchem_id'] = indication_df.stitch_id_flat.map(stitch_flat_to_pubchem)

In [65]:
indication_df

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id
0,CID100000085,C0015544,text_mention,Failure to Thrive,LLT,C0015544,Failure to thrive,85.0
1,CID100000085,C0015544,text_mention,Failure to Thrive,PT,C0015544,Failure to thrive,85.0
2,CID100000085,C0020615,text_mention,Hypoglycemia,LLT,C0020615,Hypoglycaemia,85.0
3,CID100000085,C0020615,text_mention,Hypoglycemia,PT,C0020615,Hypoglycaemia,85.0
4,CID100000085,C0022661,NLP_indication,"Kidney Failure, Chronic",LLT,C0022661,Renal failure chronic,85.0
...,...,...,...,...,...,...,...,...
30830,CID171306834,C0524910,NLP_precondition,"Hepatitis C, Chronic",PT,C0524910,Chronic hepatitis C,71306834.0
30831,CID171306834,C0741132,text_mention,Antibody test positive,LLT,C0741132,Antibody test positive,71306834.0
30832,CID171306834,C0741132,text_mention,Antibody test positive,PT,C0741132,Antibody test positive,71306834.0
30833,CID171306834,C0856536,text_mention,Philadelphia chromosome positive,LLT,C0856536,Philadelphia chromosome positive,71306834.0


In [74]:
indication_df['ChEMBL_ID'] = indication_df.apply(pubchem_id_to_InChIKey_to_ChEMBL_ID, axis=1)
indication_df

SSLError: HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Max retries exceeded with url: /unichem/rest/inchikey/UHDGCWIWMRVCDJ-UHFFFAOYSA-N (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:1000)')))

#### This is a method to get drug name given stitich ID. However, we are not using it since the database is not complete, and we can get not only the drug name but also chembl ID from UniChem database directly

In [98]:
import requests
import json
def stitch_id_to_drug_name(row):
    stitch_id = row['stitch_id_flat']
    # Set the base URL and parameters
    base_url = "http://stitch.embl.de/api/json/resolve"
    params = {
        "identifier": stitch_id #"CID054684141",  # Replace "ADD" with your desired identifier
        #"species": 9606       # Species code for humans
    }
    
    # Make the GET request
    response = requests.get(base_url, params=params)
    
    # Check if the request was successful
    if response.status_code == 200:
        # Print the response content (which will be in TSV format)
        print(response.text)
        return json.loads(response.text)[0]['preferredName']

    return stitch_id


In [59]:
stitch_id = "CID100057469"

base_url = "http://stitch.embl.de/api/json/resolve"
params = {
    "identifier": stitch_id #"CID054684141",  # Replace "ADD" with your desired identifier
    #"species": 9606       # Species code for humans
}

# Make the GET request
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    # Print the response content (which will be in TSV format)
    print(response.text)
    # return json.loads(response.text)[0]['preferredName']


[{"ncbiTaxonId":-1,"stringId":"-1.CID100057469","queryIndex":0,"annotation":"imiquimod","preferredName":"imiquimod","taxonName":"small molecule"}]


In [69]:
import requests

# Set the base URL and parameters
base_url = "http://string-db.org/api/tsv/resolve"
params = {
    "identifier": "4-aminopyridin",  # Replace "ADD" with your desired identifier
    #"species": 9606       # Species code for humans
}

# Make the GET request
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    # Print the response content (which will be in TSV format)
    print(response.text)


stringId	ncbiTaxonId	taxonName	preferredName	annotation
10116.ENSRNOP00000040830	10116	Rattus norvegicus	Kcnq2	Potassium voltage-gated channel subfamily KQT member 2; Associates with KCNQ3 to form a potassium channel with essentially identical properties to the channel underlying the native M-current, a slowly activating and deactivating potassium conductance which plays a critical role in determining the subthreshold electrical excitability of neurons as well as the responsiveness to synaptic inputs. Therefore, it is important in the regulation of neuronal excitability. KCNQ2 current is blocked by barium and tetraethylammonium whereas 4-aminopyridine and charybdotoxin have no effect on KCNQ2 c [...] 



In [55]:
response[0]

TypeError: 'Response' object is not subscriptable

In [57]:
import pandas as pd
import json

data = json.loads(response.text)
data[0]['preferredName']

'4-aminopyridine'

In [46]:
import requests

def get_drug_info_from_stitch(stitch_id):
    url = f"http://stitch.embl.de/api/json/abstracts/resolve?identifiers={stitch_id}"
    url = f"http://string-db.org/api/tsv/resolve?identifier=ADD&species=9606"
    
    response = requests.get(url)
    print(response)
    if response.status_code == 200:
        aliases = response.json()
        drug_info = {
            "drug_name": None,
            "drugbank_id": None
        }
        
        for alias in aliases:
            if 'DB' in alias['alias']:
                drug_info['drugbank_id'] = alias['alias']
            if 'PubChem Compound' in alias['source'] and drug_info['drug_name'] is None:
                drug_info['drug_name'] = alias['alias']
        
        return drug_info
    else:
        print(f"Error: Unable to retrieve data for STITCH ID {stitch_id}")
        return None

# Example usage
stitch_id = 'CID100001727'
drug_info = get_drug_info_from_stitch(stitch_id)
print(drug_info)


<Response [200]>


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [33]:
multiple_sclerosis_df = indication_df[indication_df['meddra_name'] == 'Multiple sclerosis'].query("meddra_type == 'PT'")
multiple_sclerosis_df

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id
1915,CID100001727,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,1727.0
2058,CID100001875,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,1875.0
3639,CID100002284,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2284.0
4722,CID100002554,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2554.0
6970,CID100002891,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2891.0
7220,CID100002951,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2951.0
7500,CID100003003,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,3003.0
11264,CID100003640,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,3640.0
14088,CID100004159,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,4159.0
14766,CID100004212,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,4212.0


In [82]:
multiple_sclerosis_df.merge(drugbank_map_df_2, left_on = 'pubchem_id', right_on='pubchem_cid')

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id,drugbankId,name,...,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
0,CID100001727,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,1727.0,DB06637,Dalfampridine,...,1727.0,504-24-5,CHEMBL284348,ZINC000000599985,34385,C13728,D04127,10458.0,"C0000477,C0878240,C1449659",CID000001727
1,CID100002284,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2284.0,DB00181,Baclofen,...,2284.0,1134-47-0,CHEMBL701,NaN,2972,NaN,D00241,24182.0,"C0004609,C4256145",CID000002284
2,CID100002554,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2554.0,DB14498,Potassium acetate,...,2554.0,127-08-2,CHEMBL1201058,NaN,32029,C12554,NaN,NaN,C0137984,NaN
3,CID100002554,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2554.0,DB00564,Carbamazepine,...,2554.0,298-46-4,CHEMBL108,ZINC000000004785,3387,C06868,D00252,50003659.0,C0006949,CID000002554
4,CID100004212,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,4212.0,DB01204,Mitoxantrone,...,4212.0,65271-80-9,CHEMBL58,ZINC000003794794,50729,C11195,D08224,67690.0,C0026259,CID000004212
5,CID100005487,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,5487.0,DB00697,Tizanidine,...,5487.0,51322-75-9,CHEMBL1079,ZINC000019702309,63629,C07452,D08611,50240671.0,C0146011,CID000005487


In [34]:
multiple_sclerosis_df.merge(drugbank_map_df_2, left_on="pubchem_id", right_on="pubchem_cid")

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id,drugbankId,name,...,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
0,CID100001727,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,1727.0,DB06637,Dalfampridine,...,1727.0,504-24-5,CHEMBL284348,ZINC000000599985,34385,C13728,D04127,10458.0,"C0000477,C0878240,C1449659",CID000001727
1,CID100002284,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2284.0,DB00181,Baclofen,...,2284.0,1134-47-0,CHEMBL701,NaN,2972,NaN,D00241,24182.0,"C0004609,C4256145",CID000002284
2,CID100002554,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2554.0,DB14498,Potassium acetate,...,2554.0,127-08-2,CHEMBL1201058,NaN,32029,C12554,NaN,NaN,C0137984,NaN
3,CID100002554,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2554.0,DB00564,Carbamazepine,...,2554.0,298-46-4,CHEMBL108,ZINC000000004785,3387,C06868,D00252,50003659.0,C0006949,CID000002554
4,CID100004212,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,4212.0,DB01204,Mitoxantrone,...,4212.0,65271-80-9,CHEMBL58,ZINC000003794794,50729,C11195,D08224,67690.0,C0026259,CID000004212
5,CID100005487,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,5487.0,DB00697,Tizanidine,...,5487.0,51322-75-9,CHEMBL1079,ZINC000019702309,63629,C07452,D08611,50240671.0,C0146011,CID000005487


In [19]:
mapped_indication_df = indication_df.merge(drugbank_map_df_2, left_on='stitch_id_flat', right_on='stitch_id')
mapped_indication_df

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id,drugbankId,name,...,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
0,CID100000175,C0009450,text_mention,Communicable Diseases,LLT,C0009450,Infection,175.0,DB14511,Acetate,...,NaN,71-50-1,NaN,NaN,30089,NaN,NaN,50159793.0,NaN,CID100000175
1,CID100000175,C0009450,text_mention,Communicable Diseases,PT,C0009450,Infection,175.0,DB14511,Acetate,...,NaN,71-50-1,NaN,NaN,30089,NaN,NaN,50159793.0,NaN,CID100000175
2,CID100000175,C0020625,NLP_indication,Hyponatremia,LLT,C0020625,Hyponatraemia,175.0,DB14511,Acetate,...,NaN,71-50-1,NaN,NaN,30089,NaN,NaN,50159793.0,NaN,CID100000175
3,CID100000175,C0020625,NLP_indication,Hyponatremia,PT,C0020625,Hyponatraemia,175.0,DB14511,Acetate,...,NaN,71-50-1,NaN,NaN,30089,NaN,NaN,50159793.0,NaN,CID100000175
4,CID100000222,C0220983,text_mention,Metabolic alkalosis,LLT,C0220983,Metabolic alkalosis,222.0,DB11118,Ammonia,...,NaN,7664-41-7,NaN,NaN,NaN,NaN,NaN,NaN,C0002607,CID100000222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5175,CID154687131,C0040592,text_mention,Trachoma,PT,C0040592,Trachoma,54687131.0,DB00256,Lymecycline,...,54707177.0,992-21-2,CHEMBL2103929,ZINC000053682936,59040,NaN,D06884,NaN,C0024200,CID154687131
5176,CID154687131,C0149778,text_mention,Soft Tissue Infections,LLT,C0149778,Soft tissue infection,54687131.0,DB00256,Lymecycline,...,54707177.0,992-21-2,CHEMBL2103929,ZINC000053682936,59040,NaN,D06884,NaN,C0024200,CID154687131
5177,CID154687131,C0149778,text_mention,Soft Tissue Infections,PT,C0149778,Soft tissue infection,54687131.0,DB00256,Lymecycline,...,54707177.0,992-21-2,CHEMBL2103929,ZINC000053682936,59040,NaN,D06884,NaN,C0024200,CID154687131
5178,CID154687131,C1112709,text_mention,non-gonococcal urethritis (NGU),LLT,C1112709,Unspecified non-gonococcal urethritis (NGU),54687131.0,DB00256,Lymecycline,...,54707177.0,992-21-2,CHEMBL2103929,ZINC000053682936,59040,NaN,D06884,NaN,C0024200,CID154687131


In [21]:
mapped_indication_df[mapped_indication_df['meddra_name'] == 'Multiple sclerosis']

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id,drugbankId,name,...,pubchem_cid,cas_num,chembl_id,zinc_id,chebi_id,kegg_cid,kegg_id,bindingDB_id,UMLS_cuis,stitch_id
958,CID100002891,C0026769,text_mention,Multiple Sclerosis,LLT,C0026769,Multiple sclerosis,2891.0,DB00115,Cyanocobalamin,...,70678590.0,68-19-9,CHEMBL2110563,NaN,17439,C02823,D00166,NaN,C0042845,CID100002891
959,CID100002891,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2891.0,DB00115,Cyanocobalamin,...,70678590.0,68-19-9,CHEMBL2110563,NaN,17439,C02823,D00166,NaN,C0042845,CID100002891
996,CID100002951,C0026769,text_mention,Multiple Sclerosis,LLT,C0026769,Multiple sclerosis,2951.0,DB01219,Dantrolene,...,6914273.0,7261-97-4,CHEMBL1201288,ZINC000002568036,4317,C06939,D02347,50198767.0,C0010976,CID100002951
997,CID100002951,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,2951.0,DB01219,Dantrolene,...,6914273.0,7261-97-4,CHEMBL1201288,ZINC000002568036,4317,C06939,D02347,50198767.0,C0010976,CID100002951
4221,CID104479097,C0026769,text_mention,Multiple Sclerosis,LLT,C0026769,Multiple sclerosis,4479097.0,DB00200,Hydroxocobalamin,...,70678542.0,13422-51-0,CHEMBL2103737,NaN,27786,C08230,D01027,NaN,"C0003663,C0020316,C4256142",CID104479097
4222,CID104479097,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis,4479097.0,DB00200,Hydroxocobalamin,...,70678542.0,13422-51-0,CHEMBL2103737,NaN,27786,C08230,D01027,NaN,"C0003663,C0020316,C4256142",CID104479097


In [9]:
filtered_pubchem = drugbank_map_df[drugbank_map_df['drugbank_id'] == 'DB00115']

In [10]:
filtered_pubchem.merge(indication_df)

,drugbank_id,pubchem_id,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name


In [11]:
indication_df[indication_df['pubchem_id'] == int(17439)]

,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name,pubchem_id


In [30]:
drugbank_df[drugbank_df['drugbank_name'] == 'Cyanocobalamin']

,drugbank_id,drugbank_name
109,DB00115,Cyanocobalamin


In [44]:
drugbank_df

,drugbank_id,drugbank_name
0,DB00001,Lepirudin
1,DB00002,Cetuximab
2,DB00003,Dornase alfa
3,DB00004,Denileukin diftitox
4,DB00005,Etanercept
...,...,...
7754,DB09023,Benactyzine
7755,DB09024,Follitropin Alpha
7756,DB09026,Aliskiren
7757,DB09028,Cytisine


In [45]:
drugbank_map_df

,drugbank_id,pubchem_id
0,DB00014,11980055
1,DB00014,11981235
2,DB00014,11982741
3,DB00014,16052011
4,DB00014,23581804
...,...,...
204704,DB09028,74070157
204705,DB09028,74834862
204706,DB09028,77513518
204707,DB09028,87355970


In [41]:
indication_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30835 entries, 0 to 30834
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   stitch_id_flat        30835 non-null  object 
 1   umls_cui_from_label   30835 non-null  object 
 2   method                30835 non-null  object 
 3   concept_name          30835 non-null  object 
 4   meddra_type           30794 non-null  object 
 5   umls_cui_from_meddra  30794 non-null  object 
 6   meddra_name           30835 non-null  object 
 7   pubchem_id            30835 non-null  float64
dtypes: float64(1), object(7)
memory usage: 1.9+ MB


In [23]:
indication_df = drugbank_df.merge(drugbank_map_df.merge(indication_df))
indication_df = indication_df.query("meddra_type == 'PT'")
indication_df.head(2)

,drugbank_id,drugbank_name,pubchem_id,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name
1,DB00014,Goserelin,47725,CID100047725,C0002871,text_mention,Anemia,PT,C0002871,Anaemia
3,DB00014,Goserelin,47725,CID100047725,C0006142,NLP_indication,Malignant neoplasm of breast,PT,C0006142,Breast cancer


In [24]:
indication_df[indication_df['drugbank_id'] == 'DB00115']

,drugbank_id,drugbank_name,pubchem_id,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name


In [25]:
indication_df[indication_df['meddra_name'] == 'Multiple sclerosis']

,drugbank_id,drugbank_name,pubchem_id,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name
604,DB00181,Baclofen,2284,CID100002284,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
6385,DB00443,Betamethasone,3003,CID100003003,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
9221,DB00564,Carbamazepine,2554,CID100002554,C0026769,NLP_indication,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
10580,DB00620,Triamcinolone,5544,CID100005544,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
11048,DB00635,Prednisone,4900,CID100004900,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
12222,DB00697,Tizanidine,5487,CID100005487,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
13226,DB00741,Hydrocortisone,3640,CID100003640,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
15500,DB00860,Prednisolone,4894,CID100004894,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
17142,DB00959,Methylprednisolone,4159,CID100004159,C0026769,text_mention,Multiple Sclerosis,PT,C0026769,Multiple sclerosis
22470,DB01204,Mitoxantrone,4212,CID100004212,C0026769,NLP_precondition,Multiple Sclerosis,PT,C0026769,Multiple sclerosis


In [77]:
indication_df[indication_df['meddra_name'] == 'Multiple sclerosis'].drugbank_name.tolist()

['Baclofen',
 'Betamethasone',
 'Carbamazepine',
 'Triamcinolone',
 'Prednisone',
 'Tizanidine',
 'Hydrocortisone',
 'Prednisolone',
 'Methylprednisolone',
 'Mitoxantrone',
 'Dantrolene',
 'Dexamethasone',
 'FTY 720',
 'Dalfampridine',
 '(11alpha,14beta)-11,17,21-trihydroxypregn-4-ene-3,20-dione',
 'Fingolimod']

### Question left to resolve: some drugs don't appear as indication even though they appear on the webpage: cyanocobalamin for side effect Multiple sclerosis

In [48]:
indication_df[indication_df['drugbank_name'].str.contains('Cevimeline')]

,drugbank_id,drugbank_name,pubchem_id,stitch_id_flat,umls_cui_from_label,method,concept_name,meddra_type,umls_cui_from_meddra,meddra_name
712,DB00185,Cevimeline,2684,CID100002684,C0043352,NLP_indication,Xerostomia,PT,C0043352,Dry mouth
714,DB00185,Cevimeline,2684,CID100002684,C1527336,NLP_indication,Sjogren's Syndrome,PT,C1527336,Sjogren's syndrome


In [17]:
# Multiple Sclerosis indications
indication_df.query("umls_cui_from_meddra == 'C0026769'").drugbank_name.tolist()

['Baclofen',
 'Betamethasone',
 'Carbamazepine',
 'Triamcinolone',
 'Prednisone',
 'Tizanidine',
 'Hydrocortisone',
 'Prednisolone',
 'Methylprednisolone',
 'Mitoxantrone',
 'Dantrolene',
 'Dexamethasone',
 'FTY 720',
 'Dalfampridine',
 '(11alpha,14beta)-11,17,21-trihydroxypregn-4-ene-3,20-dione',
 'Fingolimod']

In [22]:
# Save indications
# indication_df.to_csv('data/indications.tsv', sep='\t', index=False)

In [23]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import datetime
import ast


In [24]:
from chembl_webresource_client.new_client import new_client

# Define a function to retrieve DrugBank IDs based on ChEMBL IDs
def get_drugbank_id(chembl_id):
    # Create a new ChEMBL client
    chembl = new_client

    # Search for the molecule based on ChEMBL ID
    molecule = chembl.molecule.filter(chembl_id=chembl_id) #.only(['drugbank_id'])

    # Extract DrugBank ID if available
    print(molecule)
    drugbank_id = molecule[0]['drugbank_id'] if molecule else None

    return drugbank_id

# Example usage
chembl_id = 'CHEMBL3039597'  # Replace this with your ChEMBL ID
drugbank_id = get_drugbank_id(chembl_id)
if drugbank_id:
    print(f"The DrugBank ID for ChEMBL ID {chembl_id} is: {drugbank_id}")
else:
    print(f"No DrugBank ID found for ChEMBL ID {chembl_id}")


[{'atc_classifications': ['S01AA11', 'S02AA14', 'J01GB03', 'S03AA06', 'D06AX07'], 'availability_type': 1, 'biotherapeutic': None, 'black_box_warning': 1, 'chebi_par_id': None, 'chemical_probe': 0, 'chirality': -1, 'cross_references': [{'xref_id': '4265', 'xref_name': 'gentamicin', 'xref_src': 'DrugCentral'}], 'dosed_ingredient': False, 'first_approval': 1970, 'first_in_class': 0, 'helm_notation': None, 'indication_class': 'Antibacterial', 'inorganic_flag': 0, 'max_phase': '4.0', 'molecule_chembl_id': 'CHEMBL3039597', 'molecule_hierarchy': {'active_chembl_id': 'CHEMBL3039597', 'molecule_chembl_id': 'CHEMBL3039597', 'parent_chembl_id': 'CHEMBL3039597'}, 'molecule_properties': {'alogp': None, 'aromatic_rings': None, 'cx_logd': None, 'cx_logp': None, 'cx_most_apka': None, 'cx_most_bpka': None, 'full_molformula': 'C60H123N15O21', 'full_mwt': '1390.73', 'hba': None, 'hba_lipinski': None, 'hbd': None, 'hbd_lipinski': None, 'heavy_atoms': None, 'molecular_species': None, 'mw_freebase': '1390.7

KeyError: 'drugbank_id'

In [25]:
activities = new_client.activity

In [ ]:
chem_id = 'CHEMBL3039597'
res = activities.filter(molecule_chembl_id=chem_id)
target_list = []
for element in res:
    print(element)
    # if element['type'] == 'IC50':
    #     if element['standard_value']:
    #         if float(element['standard_value']) <= 1000:
    #             target_id = element['target_chembl_id']
    #             target_name = element['target_pref_name']
    #             target_list.append(target_id)
    #             if target_id not in target_id_name_dict.keys():
    #                 target_id_name_dict[target_id] = target_name
    #                 temp = pd.DataFrame([[target_id, target_name]],columns=['ChEBML ID', 'Name'])
    #                 target_id_name_df = pd.concat([target_id_name_df, temp])
# drug_dict[chem_id] = target_list

In [ ]:
molecule = new_client.molecule

In [21]:
mols = molecule.filter(

SyntaxError: incomplete input (52802753.py, line 1)

In [ ]:
mols = molecule.filter(molecule_chembl_id__icontains=chem_id)

# UniChem Cross References

In [1]:
import requests

def get_drugbank_id_from_chembl(chembl_id):
    # UniChem API endpoint for ChEMBL to DrugBank mappings
    api_url = "https://www.ebi.ac.uk/unichem/api/v1/connectivity"
    todo = {"compound": chembl_id, "searchComponents": False,  "sourceID": 1,  "type": 'sourceID'}

    try:
        # Make GET request to UniChem API
        response = requests.post(api_url, json=todo)
        response.raise_for_status()  # Raise an exception for HTTP errors

        # Parse JSON response
        data = response.json()
        # print(data)
        # Extract DrugBank ID from response
        drugbank_id = None
        # for mapping in data:
        #     if mapping['src_id'] == 22:  # DrugBank source ID
        #         drugbank_id = mapping['dest_compound_id']
        #         break
        if data['response'] == 'Not found':
            return None
        for element in data['sources']:
            if element['comparison']['stereoType'] == True and element['longName'] == 'DrugBank':
                drugbank_id = element['compoundId']

        return drugbank_id

    except requests.exceptions.RequestException as e:
        print("Error fetching data:", e)
        return None

# Example usage
chembl_id = "CHEMBL1622"
drugbank_id = get_drugbank_id_from_chembl(chembl_id)
if drugbank_id:
    print(f"The DrugBank ID corresponding to ChEMBL ID {chembl_id} is {drugbank_id}.")
else:
    print(f"No DrugBank ID found for ChEMBL ID {chembl_id}.")


The DrugBank ID corresponding to ChEMBL ID CHEMBL1622 is DB00158.


In [ ]:
import requests
api_url = "https://www.ebi.ac.uk/unichem/api/v1/connectivity"
todo = {"compound": "CHEMBL2108675", "searchComponents": False,  "sourceID": 1,  "type": 'sourceID'}
response = requests.post(api_url, json=todo)
data = response.json()
response.json()

In [ ]:
for element in data['sources']:
    if element['comparison']['stereoType'] == True and element['longName'] == 'DrugBank':
        print(element['compoundId'])

In [ ]:
import requests
api_url = "https://jsonplaceholder.typicode.com/todos"
todo = {"userId": 1, "title": "Buy milk", "completed": False}
response = requests.post(api_url, json=todo)
response.json()


response.status_code